# 11 — Preprocessing & leakage

*Module 00 · Getting Started — notebook 11 of 11*

The last step before the methods: getting features ready. We scale them, encode the non-numeric ones, and — the part that catches everyone — we do it **without letting the test set leak into training**. This notebook settles two debts the module has carried since notebooks 02 and 04.

**Prerequisites:** 04 (the train/test split and the idea of leakage), 05 (nearest centroid and its scale-sensitivity), 10 (cross-validation).

**What you'll be able to do**
- **Standardize** features, and see why notebook 05's distance-based classifier was scale-sensitive.
- **One-hot encode** a categorical feature.
- Keep every preprocessing step **fit on the training data only**, with a `Pipeline`.
- Recognize **data leakage**, and watch it inflate a score that should not exist.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import NearestCentroid
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from ml_course import colors, datasets, viz

np.random.seed(0)
viz.use_course_style()

# The module-00 two-feature binary subset (Adelie vs Gentoo), same split as notebook 05.
X, y = datasets.penguins_xy()
FEATURES = datasets.PENGUINS_FEATURES
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)
print(f'train {len(X_train)}   test {len(X_test)}   features {FEATURES}')

## Two debts to settle

This module left two promises open.

- In notebook 05, our nearest-centroid classifier measured plain Euclidean distance on **raw** bill and flipper millimetres — and we flagged (notebooks 02 and 05) that this makes it **scale-sensitive**: the feature with the larger numeric range quietly dominates the distance.
- In notebook 04, we named **leakage** — letting information from the test set sneak into training — and promised to show how preprocessing can cause it.

We pay off both here. First, scaling.

## Standardizing features

Look at our two features: bill length spans roughly five millimetres of variation, flipper length about fifteen. Euclidean distance adds up squared differences across features, so a feature with a larger spread contributes more to every distance — not because it matters more, but because its numbers are bigger. A model that decides by distance (like nearest centroid) then leans on flipper length almost by accident.

**Standardization** removes that accident. For each feature we subtract its mean and divide by its standard deviation, computed on the **training** data:

`z = (x - mean) / std`

Every feature is now centred at zero with a spread of one, so all of them get an equal say. The units become "standard deviations from the mean."

In [ ]:
mean = X_train.mean()
std = X_train.std()
print('training-set mean per feature:')
print(mean.round(2).to_string())
print('\ntraining-set std per feature:')
print(std.round(2).to_string())

X_train_z = (X_train - mean) / std
print('\nafter standardizing the training set, means are ~0 and stds are ~1:')
print(X_train_z.mean().round(3).to_string())
print(X_train_z.std().round(3).to_string())

In [ ]:
# Fit on arrays so the dense grid matches the model's inputs (no feature-name warning).
X_train_arr, X_test_arr = X_train.to_numpy(), X_test.to_numpy()

raw_nc = NearestCentroid().fit(X_train_arr, y_train)
std_nc = make_pipeline(StandardScaler(), NearestCentroid()).fit(X_train_arr, y_train)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
viz.plot_decision_boundary(raw_nc, X_train, y_train, resolution=200, ax=axes[0])
axes[0].set_title('raw features')
viz.plot_decision_boundary(std_nc, X_train, y_train, resolution=200, ax=axes[1])
axes[1].set_title('standardized (inside a pipeline)')
plt.show()

print(f'test accuracy  -  raw: {raw_nc.score(X_test_arr, y_test):.3f}   '
      f'standardized: {std_nc.score(X_test_arr, y_test):.3f}')

### Read the figure

Same data, same classifier — only the feature scaling differs, and the decision boundary visibly **rotates** (by about 52° in the figure's millimetre coordinates). On the raw features the boundary is pulled by flipper length (the wider-spread feature); after standardizing, bill and flipper count equally and the boundary tilts to reflect that. The accuracy hardly moves here (these two species sit far apart, so both versions get nearly everything right — and on this one split the raw version even edges ahead by luck), but the **model itself genuinely changed**. That is the scale-sensitivity notebook 05 warned about, now in plain view. To compare the two fairly — not on one lucky split — we turn to cross-validation.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
raw_cv = cross_val_score(NearestCentroid(), X, y, cv=cv).mean()
std_cv = cross_val_score(make_pipeline(StandardScaler(), NearestCentroid()), X, y, cv=cv).mean()
print(f'5-fold CV accuracy  -  raw: {raw_cv:.4f}   standardized: {std_cv:.4f}')

### Read the output

Averaged over five folds, the standardized version comes out slightly ahead (about 0.993 vs 0.989). The margin is small **because these classes are so well separated** — scaling cannot add much when almost everything is already correct. On harder problems, or features whose scales differ wildly, standardizing is often decisive; it is the principled default for any method that measures distances, which is why notebooks like k-nearest-neighbours and support vector machines will lean on it.

## Encoding categorical features

Scaling handles numbers. But real datasets carry **text** too. The full penguins table we fetched in notebook 01 has an `island` column (`Biscoe`, `Dream`, `Torgersen`) and a `sex` column — categories, not quantities. A model cannot multiply "Biscoe" by a weight.

The standard fix for an **unordered** category is **one-hot encoding**: replace the single text column with several 0/1 columns, one per category. A Biscoe penguin gets a 1 in the `island_Biscoe` column and 0 in the others.

In [ ]:
full = datasets.load_penguins_full()
print('island counts:')
print(full['island'].value_counts().to_string())

# One-hot by hand: one 0/1 column per island.
island_dummies = pd.get_dummies(full['island'], prefix='island').astype(int)
print('\none-hot columns:', list(island_dummies.columns))
island_dummies.head()

In [ ]:
# The same idea with scikit-learn's encoder, fit on the TRAINING islands only.
full_bin = (
    full[full['species'].isin(['Adelie', 'Gentoo'])]
    .dropna(subset=[*FEATURES, 'island'])
    .reset_index(drop=True)
)
isl_train, isl_test = train_test_split(
    full_bin[['island']], test_size=0.25, random_state=0, stratify=full_bin['species']
)
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False).fit(isl_train)
print('categories learned from the training islands:', list(encoder.categories_[0]))
print(f'{len(isl_test)} test islands -> {encoder.transform(isl_test).shape} array of 0/1 columns')
print('\none-hot encoding of each island:')
print(encoder.transform(pd.DataFrame({'island': ['Biscoe', 'Dream', 'Torgersen']})))

### Read the output

One text column became three 0/1 columns — one per island — and scikit-learn's `OneHotEncoder` learned its list of categories from the **training data only**. That matters: if a category never seen in training turns up at test time, `handle_unknown="ignore"` encodes it as all-zeros instead of crashing or, worse, quietly refitting on the test set.

Why one-hot rather than mapping the islands to 1, 2, 3? Because that would invent an **order** (and spacing) the categories do not have — it would tell the model Torgersen is "three times" Biscoe, which is meaningless. One column per category imposes no such fiction.

## One object that does it all: the Pipeline

Numeric features want scaling; categorical features want one-hot encoding. Different columns need different treatment. scikit-learn's **`ColumnTransformer`** routes each group of columns to its own transform (you give it triples — a name, the transform, and the columns it applies to), and a **`Pipeline`** chains that preprocessing together with the classifier into a single object.

The payoff is not tidiness — it is safety. When this one object is handed to cross-validation, **every transform is fit on the training fold alone**, automatically, on every fold. The scaler's mean, the encoder's categories: all learned without ever touching the held-out data. That is exactly the discipline that prevents leakage, and the `Pipeline` makes it the default rather than something to remember.

In [ ]:
Xc = full_bin[[*FEATURES, 'island']]
yc = full_bin['species']

preprocessor = ColumnTransformer([
    ('scale', StandardScaler(), FEATURES),
    ('encode', OneHotEncoder(handle_unknown='ignore'), ['island']),
])
clf = Pipeline([('prep', preprocessor), ('model', LogisticRegression(max_iter=1000))])

score = cross_val_score(clf, Xc, yc, cv=cv).mean()
print(f'pipeline 5-fold CV accuracy (scaled features + one-hot island): {score:.4f}')

### Read the output

One object scaled the two numeric features, one-hot encoded the island, fitted a classifier on the combination, and did all of that afresh inside each of the five folds — no held-out row ever influenced a scaler or an encoder. This is the standard, leakage-safe way to preprocess. (Island turns out to be informative for telling Adélies from Gentoos, which is realistic — but the lesson here is the plumbing, done safely.)

## When preprocessing leaks

Here is the rule the `Pipeline` was quietly enforcing: **fit every preprocessing step on the training data only**. Fit it on all the data — before the split, or on the whole set "to save time" — and information about the test set flows into the model. The reported score then measures something that will not survive contact with genuinely new data.

On our penguins this is hard to demonstrate, because they are so clean. Let us look anyway, then build a case where the leak is impossible to miss.

In [ ]:
# Fit the scaler on TRAIN only (correct) vs on ALL the data (leaky), then score the test set.
correct_model = make_pipeline(StandardScaler(), NearestCentroid()).fit(X_train, y_train)
correct = correct_model.score(X_test, y_test)

scaler_all = StandardScaler().fit(X)  # leak: this has seen the test rows too
nc_leaky = NearestCentroid().fit(scaler_all.transform(X_train), y_train)
leaky = (nc_leaky.predict(scaler_all.transform(X_test)) == y_test.to_numpy()).mean()

print(f'correct (scaler fit on train): {correct:.4f}    leaky (scaler fit on all): {leaky:.4f}')

### Read the output

Almost no difference. Standardizing on all the data versus on the training data alone barely changes the test score here, because the scaler only learns a mean and a standard deviation, and those are nearly identical whether or not the 69 test penguins are included. A small, forgivable-looking leak — which is exactly what makes it dangerous: on clean data it hides, so people stop guarding against it. The harm shows up when the preprocessing learns *much more* from the data — for example, when it **selects features**.

## The wrong way to cross-validate

This is a classic cautionary tale (from *The Elements of Statistical Learning*). We build a deliberately hopeless dataset: 100 examples, **1000 features that are pure random noise**, and labels assigned at random. Nothing predicts anything — an honest classifier should score about 50%, a coin flip.

Now we do something tempting: keep only the 20 features most correlated with the label, then cross-validate a classifier on them. The only question is **when** we choose those 20 features — using all the data up front, or inside each fold.

In [ ]:
rng = np.random.default_rng(0)
n, p, k = 100, 1000, 20
Z = rng.standard_normal((n, p))      # 1000 pure-noise features
labels = rng.integers(0, 2, size=n)  # labels independent of Z -> honest accuracy is ~0.5

# WRONG: pick the 20 features most correlated with the label on ALL data, then cross-validate.
f_scores, _ = f_classif(Z, labels)
top_k = np.argsort(f_scores)[-k:]
wrong = cross_val_score(LogisticRegression(max_iter=2000), Z[:, top_k], labels, cv=cv).mean()

# RIGHT: do the selection INSIDE each fold (a pipeline), so it never sees that fold's held-out rows.
right = cross_val_score(
    make_pipeline(SelectKBest(f_classif, k=k), LogisticRegression(max_iter=2000)), Z, labels, cv=cv
).mean()

fig, ax = plt.subplots(figsize=(6.5, 4.5))
bars = ax.bar(
    ['select on ALL data,\nthen cross-validate', 'select INSIDE\neach fold'],
    [wrong, right],
    color=[colors.COLORS['highlight'], colors.COLORS['model']],
    edgecolor=colors.COLORS['text'], linewidth=0.6,
)
for bar, value in zip(bars, [wrong, right], strict=True):
    ax.text(bar.get_x() + bar.get_width() / 2, value, f'{value:.2f}', ha='center', va='bottom',
            color=colors.COLORS['text'])
ax.axhline(0.5, color=colors.COLORS['muted'], linestyle='--', linewidth=1,
           label='honest accuracy (chance)')
ax.set_ylabel('5-fold CV accuracy')
ax.set_ylim(0, 1.0)
ax.legend()
ax.grid(False)
plt.show()
print(f'pure-noise features -> honest accuracy ~0.5.   wrong={wrong:.3f}   right={right:.3f}')

### Read the figure

The left bar is a disaster dressed as a triumph: **about 85% accuracy on data that is pure noise**, where the truth is 50%. The leak is in the selection. By scoring all 1000 features against the label using the *whole* dataset, we let the choice peek at the very rows each fold would later hold out — so the "best" 20 features are the ones that happen to match those rows by chance. Cross-validation cannot catch the cheat, because the cheat already happened before it started.

The right bar does the identical selection **inside each fold**, on the training rows only. Accuracy falls to about chance — the honest answer. Nothing changed but the *timing* of the preprocessing, and that was the difference between 85% and the truth. A `Pipeline` with the selector inside it makes the right way the natural way.

## Your turn

1. You standardize your features using the mean and standard deviation of the **whole** dataset, then split into train and test. Name the leak, and the one-line fix.
2. You have a `colour` feature with values `red`, `green`, `blue`. Why is one-hot encoding safer than mapping them to 1, 2, 3?
3. A colleague reports 95% accuracy, but mentions they scaled and selected features on the full dataset before splitting. What has probably happened, and what would you ask them to redo?

## What you built

You can now prepare features the honest way:

- You **standardized** features by hand and saw it rotate notebook 05's decision boundary — the scale-sensitivity we had only named before.
- You **one-hot encoded** a categorical feature, and saw why an unordered category must not become 1, 2, 3.
- You combined scaling and encoding in a **`ColumnTransformer`** inside a **`Pipeline`**, so every step is fit on the training fold only.
- You watched **leakage** turn pure noise into 85% accuracy — and saw the `Pipeline` undo it.

**Vocabulary added:** standardization (z-score), one-hot encoding, nominal vs ordinal, `ColumnTransformer`, `Pipeline`, fit-on-train-only, data leakage.

This closes module 00. You now have the full honest workflow — look at the data, split it, fit a model, evaluate it, validate your choices, and prepare features without leaking — the foundation every method in the chapters ahead is built on.

## Going further (optional)

- **Other leaks.** The same trap appears whenever preprocessing learns from data: filling in missing values with a column mean computed on the full set (the full penguins table's `sex` column has gaps), or encoding a category by its average target value. The rule is always the same — fit on train, then apply to test.
- **One-hot and redundancy.** The columns one-hot creates sum to one, so they are linearly dependent; a regularized model (like the logistic regression here) handles that, while some setups drop one column to break the redundancy.
- **Nested cross-validation.** When you both *select* a model and *estimate* its performance, a single CV loop can leak the selection into the estimate; nesting a second loop inside the first keeps them honest.
- **A rule of thumb.** "Fit on train; transform train and test." If you can say that sentence about every step of your preparation, you have not leaked.

None of this is needed to start the next chapter; it is here for the curious.

## References

- G. James, D. Witten, T. Hastie, R. Tibshirani, *An Introduction to Statistical Learning*, 2nd ed., Springer, 2021 — chapter 5 (resampling) and the data-preparation discussion. DOI: 10.1007/978-1-0716-1418-1
- T. Hastie, R. Tibshirani, J. Friedman, *The Elements of Statistical Learning*, 2nd ed., Springer, 2009 — §7.10.2, "the wrong and right way to do cross-validation." DOI: 10.1007/978-0-387-84858-7
- scikit-learn developers, *Common pitfalls and recommended practices* (data leakage), and the `Pipeline` / `ColumnTransformer` user guide. https://scikit-learn.org/stable/common_pitfalls.html
- A. Géron, *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, 3rd ed., O'Reilly, 2022 — chapter 2 (transformation pipelines).

---
Previous: **10 — Validating honestly: cross-validation** · Next: **module 01 — k-Nearest Neighbours**